In [1]:
# Load the trained outputs for mogonet
import os
from mogonet.prepare_trte_data import prepare_trte_data
from mogonet.test_mogonet import test_mogonet
from mogonet.train_mogonet import train_mogonet
from mogonet.utils import load_model_dict
import pickle
import pandas as pd

In [2]:
import torch
import torch.nn.functional as F
# Custom imports from the package
from mogonet.utils import (
    one_hot_tensor, cal_sample_weight, 
    gen_adj_mat_tensor, gen_test_adj_mat_tensor, 
    cal_adj_mat_parameter, get_view_list, check_adj_param_size
)

from mogonet.prepare_trte_data import prepare_trte_data
from mogonet.models import init_model_dict, init_optim

def gen_trte_adj_mat(data_tr_list, data_trte_list, trte_idx, adj_parameter):
    adj_metric = "cosine" # cosine distance
    adj_train_list = []
    adj_test_list = []
    for i in range(len(data_tr_list)):
        adj_parameter_adaptive = cal_adj_mat_parameter(adj_parameter, data_tr_list[i], adj_metric)
        adj_train_list.append(gen_adj_mat_tensor(data_tr_list[i], adj_parameter_adaptive, adj_metric))
        adj_test_list.append(gen_test_adj_mat_tensor(data_trte_list[i], trte_idx, adj_parameter_adaptive, adj_metric))
    
    return adj_train_list, adj_test_list


# Helper for epoch training
def train_epoch(data_list, adj_list, label, one_hot_label, sample_weight, model_dict, optim_dict, train_VCDN=True):
    loss_dict = {}
    criterion = torch.nn.CrossEntropyLoss(reduction='none')
    for m in model_dict:
        model_dict[m].train()
    num_view = len(data_list)
    for i in range(num_view):
        optim_dict["C{:}".format(i+1)].zero_grad()
        ci_loss = 0
        ci = model_dict["C{:}".format(i+1)](model_dict["E{:}".format(i+1)](data_list[i],adj_list[i]))
        ci_loss = torch.mean(torch.mul(criterion(ci, label),sample_weight))
        ci_loss.backward()
        optim_dict["C{:}".format(i+1)].step()
        loss_dict["C{:}".format(i+1)] = ci_loss.detach().cpu().numpy().item()
    if train_VCDN and num_view >= 2:
        optim_dict["C"].zero_grad()
        c_loss = 0
        ci_list = []
        for i in range(num_view):
            ci_list.append(model_dict["C{:}".format(i+1)](model_dict["E{:}".format(i+1)](data_list[i],adj_list[i])))
        c = model_dict["C"](ci_list)
        c_loss = torch.mean(torch.mul(criterion(c, label),sample_weight))
        c_loss.backward()
        optim_dict["C"].step()
        loss_dict["C"] = c_loss.detach().cpu().numpy().item()

    return loss_dict


In [59]:
def test_epoch(data_list, adj_list, te_idx, model_dict):
    for m in model_dict:
        model_dict[m].eval()
    num_view = len(data_list)
    ci_list = []
    for i in range(num_view):
        ci_list.append(model_dict["C{:}".format(i+1)](model_dict["E{:}".format(i+1)](data_list[i],adj_list[i])))
    if num_view >= 2:
        c = model_dict["C"](ci_list)    
    else:
        c = ci_list[0]
    c = c[te_idx,:]
    prob = F.softmax(c, dim=1).data.cpu().numpy()
    
    return prob

In [69]:
# Lots of parameters
# Preprocessing related
gse = "GSE1"
fold = "fold_2"
label = f"{gse}-{fold}"
mogo_dir = f"debug_mogo_mod"
data_folder = f"{mogo_dir}/{gse}/{fold}"
# Test related
#model_path = f"{mogo_dir}/train/{gse}/{fold}/{label}-model.pt"
#test_input_path = f"{mogo_dir}/train/{gse}/{fold}/{label}-test_input.pkl"

# Parameters to use 
lr_e_pretrain=1e-3
lr_e=5e-4
lr_c=1e-3
num_epoch_pretrain=50
num_epoch=500
num_class=2
adj_parameter=1

In [39]:
# So technically I should run this to train a particular fold of data
ideal_model_dict, test_input = train_mogonet(data_folder, lr_e_pretrain=lr_e_pretrain, 
                                       lr_e=lr_e, lr_c=lr_c, num_epoch_pretrain=num_epoch_pretrain, num_epoch=num_epoch)

Did not provide custom view list, finding it now
Found cuda, using GPU

Pretrain GCNs...

Training...


In [70]:
# The longway to do it
# Check if view_list provided or not 
view_list = get_view_list(data_folder)
# Check if cuda used or not
cuda = True if torch.cuda.is_available() else False
if cuda:
    print("Found cuda, using GPU")
else:
    print("Cuda not found, using CPU")

# Parameters 
num_view = len(view_list)
dim_hvcdn = pow(num_class,num_view)
# ----------------------------------
# Modify LATER!!!!!!!
dim_he_list = [50] * num_view # Need a more robust way to decide this
# But it should be of length N (numbers of blocks), so might need to column number
# in each block minus some constant, such this number < column numebr of specific block
# ---------------------------------
data_tr_list, data_trte_list, trte_idx, labels_trte = prepare_trte_data(data_folder, view_list)
labels_tr_tensor = torch.LongTensor(labels_trte[trte_idx["tr"]])
onehot_labels_tr_tensor = one_hot_tensor(labels_tr_tensor, num_class)
sample_weight_tr = cal_sample_weight(labels_trte[trte_idx["tr"]], num_class)
sample_weight_tr = torch.FloatTensor(sample_weight_tr)
if cuda:
    labels_tr_tensor = labels_tr_tensor.cuda()
    onehot_labels_tr_tensor = onehot_labels_tr_tensor.cuda()
    sample_weight_tr = sample_weight_tr.cuda()

# TODO: FIX THIS AND MAKE IT MORE VERBOSE
# adj_parameter needs to be <= numples of rows in train data
adj_parameter = check_adj_param_size(adj_parameter, data_tr_list)
adj_tr_list, adj_te_list = gen_trte_adj_mat(data_tr_list, data_trte_list, trte_idx, adj_parameter)
# Feature dimension of each block in train data list
dim_list = [x.shape[1] for x in data_tr_list]
# Initialize a model dictionary to update later
# Dim_he_list could be tuneable like [some_tune_num] * num_view or different numbers of each view
# like [k1, k2, k3, ... ] or just [k, k, k, ...], to list of length is equal to num_view
model_dict = init_model_dict(num_view, num_class, dim_list, dim_he_list, dim_hvcdn)
initial_model_dict = model_dict.copy()
for m in model_dict:
    if cuda:
        model_dict[m].cuda()

print("\nPretrain GCNs...")
optim_dict = init_optim(num_view, model_dict, lr_e_pretrain, lr_c)
# Some pretraining to happen
if num_epoch_pretrain == 0:
    print("Not pretraining")
else:
    for epoch in range(num_epoch_pretrain):
        # Notice the model_dict is being changed under the hood
        # for every model_dict[m].state_dict()
        train_epoch(data_tr_list, adj_tr_list, labels_tr_tensor,
                    onehot_labels_tr_tensor, sample_weight_tr, model_dict, 
                    optim_dict, train_VCDN=False)
# Main logic to train now

print("\nTraining...")
optim_dict = init_optim(num_view, model_dict, lr_e, lr_c)
for epoch in range(num_epoch+1):
    # Notice the model_dict is being changed under the hood
    # for every model_dict[m].state_dict()
    train_epoch(data_tr_list, adj_tr_list, labels_tr_tensor,
                onehot_labels_tr_tensor, sample_weight_tr, model_dict, optim_dict)

Found cuda, using GPU

Pretrain GCNs...

Training...


In [75]:
initial_model_dict['E1'].state_dict()

OrderedDict([('gc1.weight',
              tensor([[-0.0210,  0.0483, -0.0175,  ..., -0.0556,  0.0831,  0.0232],
                      [-0.0127,  0.0730, -0.0641,  ...,  0.0016, -0.0896,  0.1339],
                      [ 0.0700, -0.0731, -0.0841,  ..., -0.0493, -0.0475, -0.0245],
                      ...,
                      [ 0.0106, -0.0023, -0.0124,  ..., -0.0225, -0.0257,  0.0125],
                      [ 0.0207, -0.0355,  0.0494,  ..., -0.0888, -0.0355, -0.0406],
                      [-0.1290, -0.0440, -0.0504,  ...,  0.0022, -0.0968,  0.0481]],
                     device='cuda:0')),
             ('gc1.bias',
              tensor([-0.0078,  0.0034, -0.0155, -0.0059, -0.0194,  0.0019, -0.0300, -0.0154,
                      -0.0193, -0.0005,  0.0044,  0.0183, -0.0072,  0.0154, -0.0274, -0.0298,
                       0.0043, -0.0125,  0.0209,  0.0031, -0.0118, -0.0337, -0.0165, -0.0042,
                      -0.0206, -0.0109, -0.0114, -0.0066, -0.0252, -0.0194, -0.0104, -0.0271

In [74]:
model_dict['E1'].state_dict()

OrderedDict([('gc1.weight',
              tensor([[-0.0210,  0.0483, -0.0175,  ..., -0.0556,  0.0831,  0.0232],
                      [-0.0127,  0.0730, -0.0641,  ...,  0.0016, -0.0896,  0.1339],
                      [ 0.0700, -0.0731, -0.0841,  ..., -0.0493, -0.0475, -0.0245],
                      ...,
                      [ 0.0106, -0.0023, -0.0124,  ..., -0.0225, -0.0257,  0.0125],
                      [ 0.0207, -0.0355,  0.0494,  ..., -0.0888, -0.0355, -0.0406],
                      [-0.1290, -0.0440, -0.0504,  ...,  0.0022, -0.0968,  0.0481]],
                     device='cuda:0')),
             ('gc1.bias',
              tensor([-0.0078,  0.0034, -0.0155, -0.0059, -0.0194,  0.0019, -0.0300, -0.0154,
                      -0.0193, -0.0005,  0.0044,  0.0183, -0.0072,  0.0154, -0.0274, -0.0298,
                       0.0043, -0.0125,  0.0209,  0.0031, -0.0118, -0.0337, -0.0165, -0.0042,
                      -0.0206, -0.0109, -0.0114, -0.0066, -0.0252, -0.0194, -0.0104, -0.0271

In [62]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [67]:
a = "asdasdasd"

In [68]:
a

'asdasdasd'

In [66]:
test_interval = 50
view_list = get_view_list(data_folder)
# Check if cuda used or not
cuda = True if torch.cuda.is_available() else False
if cuda:
    print("Found cuda, using GPU")
else:
    print("Cuda not found, using CPU")
# Parameters 
num_view = len(view_list)
dim_hvcdn = pow(num_class,num_view)
data_tr_list, data_trte_list, trte_idx, labels_trte = prepare_trte_data(data_folder, view_list)
labels_tr_tensor = torch.LongTensor(labels_trte[trte_idx["tr"]])
onehot_labels_tr_tensor = one_hot_tensor(labels_tr_tensor, num_class)
sample_weight_tr = cal_sample_weight(labels_trte[trte_idx["tr"]], num_class)
sample_weight_tr = torch.FloatTensor(sample_weight_tr)
if cuda:
    labels_tr_tensor = labels_tr_tensor.cuda()
    onehot_labels_tr_tensor = onehot_labels_tr_tensor.cuda()
    sample_weight_tr = sample_weight_tr.cuda()
adj_tr_list, adj_te_list = gen_trte_adj_mat(data_tr_list, data_trte_list, trte_idx, adj_parameter)
dim_list = [x.shape[1] for x in data_tr_list]
model_dict = init_model_dict(num_view, num_class, dim_list, dim_he_list, dim_hvcdn)
for m in model_dict:
    if cuda:
        model_dict[m].cuda()

print("\nPretrain GCNs...")
optim_dict = init_optim(num_view, model_dict, lr_e_pretrain, lr_c)
for epoch in range(num_epoch_pretrain):
    train_epoch(data_tr_list, adj_tr_list, labels_tr_tensor, 
                onehot_labels_tr_tensor, sample_weight_tr, model_dict, optim_dict, train_VCDN=False)
print("\nTraining...")
optim_dict = init_optim(num_view, model_dict, lr_e, lr_c)
for epoch in range(num_epoch+1):
    train_epoch(data_tr_list, adj_tr_list, labels_tr_tensor, 
                onehot_labels_tr_tensor, sample_weight_tr, model_dict, optim_dict)
    if epoch % test_interval == 0:
        te_prob = test_epoch(data_trte_list, adj_te_list, trte_idx["te"], model_dict)
        print("\nTest: Epoch {:d}".format(epoch))
        if num_class == 2:
            print("Test ACC: {:.3f}".format(accuracy_score(labels_trte[trte_idx["te"]], te_prob.argmax(1))))
            print("Test F1: {:.3f}".format(f1_score(labels_trte[trte_idx["te"]], te_prob.argmax(1))))
            print("Test AUC: {:.3f}".format(roc_auc_score(labels_trte[trte_idx["te"]], te_prob[:,1])))
        else:
            print("Test ACC: {:.3f}".format(accuracy_score(labels_trte[trte_idx["te"]], te_prob.argmax(1))))
            print("Test F1 weighted: {:.3f}".format(f1_score(labels_trte[trte_idx["te"]], te_prob.argmax(1), average='weighted')))
            print("Test F1 macro: {:.3f}".format(f1_score(labels_trte[trte_idx["te"]], te_prob.argmax(1), average='macro')))

Found cuda, using GPU

Pretrain GCNs...

Training...

Test: Epoch 0
Test ACC: 0.700
Test F1: 0.667
Test AUC: 0.667

Test: Epoch 50
Test ACC: 0.700
Test F1: 0.667
Test AUC: 0.750

Test: Epoch 100
Test ACC: 0.700
Test F1: 0.667
Test AUC: 0.708

Test: Epoch 150
Test ACC: 0.600
Test F1: 0.000
Test AUC: 0.750

Test: Epoch 200
Test ACC: 0.600
Test F1: 0.000
Test AUC: 0.625

Test: Epoch 250
Test ACC: 0.700
Test F1: 0.667
Test AUC: 0.750

Test: Epoch 300
Test ACC: 0.700
Test F1: 0.667
Test AUC: 0.750

Test: Epoch 350
Test ACC: 0.700
Test F1: 0.667
Test AUC: 0.708

Test: Epoch 400
Test ACC: 0.600
Test F1: 0.000
Test AUC: 0.667

Test: Epoch 450
Test ACC: 0.700
Test F1: 0.667
Test AUC: 0.667

Test: Epoch 500
Test ACC: 0.700
Test F1: 0.667
Test AUC: 0.750


In [41]:
initial_model_dict["C"].state_dict()

OrderedDict([('model.0.weight',
              tensor([[-0.1170,  0.3707,  0.3038,  0.1694, -0.3253, -0.0201,  0.0125, -0.0392],
                      [ 0.1607,  0.3716,  0.1251, -0.1436,  0.2102, -0.6861,  0.2594,  0.1290],
                      [ 0.0369,  0.0484,  0.0248,  0.2235, -0.0078,  0.4464, -0.2665,  0.3007],
                      [-0.2954, -0.1857, -0.1783,  0.0431,  0.5695,  0.1046, -0.0731, -0.1909],
                      [ 0.1924,  0.6438,  0.0878,  0.2872,  0.1540,  0.3072,  0.3914,  0.0577],
                      [-0.1184, -0.4347, -0.3781, -0.0921, -0.1441,  0.0098, -0.3633,  0.8932],
                      [ 0.2294,  0.0643,  0.1168, -0.0430, -1.2629,  0.1722, -0.3797,  0.2099],
                      [-0.1916,  0.1078, -0.7342,  0.4072, -0.1818,  0.0122,  0.3748,  0.5320]],
                     device='cuda:0')),
             ('model.0.bias',
              tensor([-0.0394,  0.0234,  0.0910, -0.0576,  0.0999, -0.0930, -0.0713, -0.0292],
                     device='cuda:

In [42]:
model_dict["C"].state_dict()

OrderedDict([('model.0.weight',
              tensor([[-0.1170,  0.3707,  0.3038,  0.1694, -0.3253, -0.0201,  0.0125, -0.0392],
                      [ 0.1607,  0.3716,  0.1251, -0.1436,  0.2102, -0.6861,  0.2594,  0.1290],
                      [ 0.0369,  0.0484,  0.0248,  0.2235, -0.0078,  0.4464, -0.2665,  0.3007],
                      [-0.2954, -0.1857, -0.1783,  0.0431,  0.5695,  0.1046, -0.0731, -0.1909],
                      [ 0.1924,  0.6438,  0.0878,  0.2872,  0.1540,  0.3072,  0.3914,  0.0577],
                      [-0.1184, -0.4347, -0.3781, -0.0921, -0.1441,  0.0098, -0.3633,  0.8932],
                      [ 0.2294,  0.0643,  0.1168, -0.0430, -1.2629,  0.1722, -0.3797,  0.2099],
                      [-0.1916,  0.1078, -0.7342,  0.4072, -0.1818,  0.0122,  0.3748,  0.5320]],
                     device='cuda:0')),
             ('model.0.bias',
              tensor([-0.0394,  0.0234,  0.0910, -0.0576,  0.0999, -0.0930, -0.0713, -0.0292],
                     device='cuda:

In [21]:
ideal_model_dict["C"]

VCDN(
  (model): Sequential(
    (0): Linear(in_features=8, out_features=8, bias=True)
    (1): LeakyReLU(negative_slope=0.25)
    (2): Linear(in_features=8, out_features=2, bias=True)
  )
)